# 05 Synthèse Finale & Journal de Bord

## Objectif de ce notebook

Ce notebook constitue le document de synthèse du projet **Telemed Urgence IA**
Il résume la démarche complète, les décisions prises à chaque étape, et propose
une démonstration d'inférence de bout en bout avec interprétation lisible.

> Les analyses détaillées sont dans les notebooks `01` à `04`. Ce document
> sert de fil conducteur pour la relecture et la soutenance

## Sommaire

1. Contexte et enjeu métier
2. Démarche méthodologique et choix assumés
3. Journal de bord chronologique
4. Démonstration d'inférence avec interprétation lisible
5. Limites et perspectives

## 1. Contexte et enjeu métier

L'engorgement des services d'urgence et le développement de la télémédecine imposent
des solutions de tri rapide et fiables. Le système développé est un **outil d'aide à
la décision** — il ne remplace pas un professionnel de santé.

**Enjeu métier central** : une urgence vitale (classe 2) classée en 0 ou 1 est une
erreur bien plus grave qu'un cas non urgent classé trop haut. Toute la chaîne
d'évaluation et le choix du modèle final sont orientés par cette asymétrie de coût.

## 2. Démarche méthodologique et choix assumés

**Démarche suivie** (incrémentale, du data science vers l'industrialisation):

1. Explorer les données (`01_EDA.ipynb`)
2. Construire des pipelines de prétraitement réutilisables (`02_Preprocessing.ipynb`)
3. Comparer plusieurs modèles sur 4 scénarios de données (`03_Modelisation.ipynb`)
4. Optimiser les hyperparamètres (`04_Hyperparameter_Tuning.ipynb`)
5. Ajuster le seuil de décision pour réduire les erreurs critiques (`src/optimize_critical_errors.py`)
6. Industrialiser : API FastAPI, interface Streamlit, MLflow, Docker, CI/CD, monitoring

**Choix assumés** :

- **Le F1 pondéré ne décide pas seul.** Il sert de comparaison globale, mais le choix
  final est gouverné par le recall de la classe 2 et le nombre d'erreurs critiques
- **Pipelines scikit-learn plutôt que transformations manuelles** : garantit que les
  mêmes transformations sont appliquées à l'entraînement et à l'inférence (pas de fuite)
- **Suppression de `patient_id`** dès le chargement : identifiant non prédictif, risque RGPD.
- **Seuil de décision comme levier métier** plutôt que re-pondération de la loss
  uniquement : lisible, ajustable sans réentraînement complet.
- **TF-IDF plutôt qu'un modèle de langage lourd** : rapide, interprétable, coût
  d'inférence quasi nul, suffisant pour ce dataset.

## 3. Journal de bord chronologique

**Phase 1 — Mise en place du projet.** Structure du dépôt (`src/`, `notebooks/`,
`tests/`, `data/`, `models/`), environnement virtuel, premier commit GitHub

**Phase 2 — Exploration des données.** 10 080 lignes, 77 doublons détectés
(0.76%, jugés sans impact significatif), peu de valeurs manquantes (<0.3% par
colonne), classe 2 minoritaire à 14.64%. Conclusion : suivre explicitement le
recall de la classe 2, pas seulement l'accuracy. Les médianes des constantes
vitales confirment une progression clinique cohérente avec la gravité.

**Phase 3 — Prétraitement.** Pipelines scikit-learn pour les 4 scénarios
(`full`, `without_sensitive`, `text_only`, `clinical_only`). Validation que
chaque scénario transforme correctement les 10 080 lignes sans perte

**Phase 4 — Comparaison initiale des modèles.** Sur le scénario `full` :
LogisticRegression, RandomForest et XGBoost comparés. XGBoost gagne sur le F1
pondéré, mais LogisticRegression a déjà le meilleur recall classe 2 avant
optimisation

**Phase 5 — Comparaison des 4 scénarios.** `full` et `without_sensitive`
quasi identiques en performance — argument décisif pour retirer les variables
sensibles (sexe, zone_vie) sans perte significative. `text_only` montre que
le texte seul atteint ~91% d'accuracy. `clinical_only` est insuffisant pour
détecter la classe 2 (recall ~0.74-0.79)

**Phase 6 — Tuning des hyperparamètres.** RandomizedSearchCV sur 4 modèles.
Confirmation que LogisticRegression reste compétitif après tuning.

**Phase 7 — Optimisation des erreurs critiques.** Ajustement du seuil de
décision de la classe 2 sur 3 modèles (LogReg, RandomForest, XGBoost).
**XGBoost avec seuil 0.01** ressort comme le meilleur compromis : recall
classe 2 de 98.98%, seulement 3 erreurs critiques sur le jeu de test, tout en
conservant le meilleur F1 pondéré parmi les modèles atteignant ce niveau de
sécurité.

**Phase 8 — Interprétation du texte.** Identification des termes les plus
caractéristiques par classe via TF-IDF (après filtrage des stopwords) :
"respiratoire", "convulsions", "plaie" pour la classe 2 ; "traumatisme",
"saignement persistant" pour la classe 1 ; "traitement", "activité sportive"
pour la classe 0

**Phase 9 — Industrialisation.** API FastAPI sécurisée (clé API pour
`/retrain`), interface Streamlit, feedback utilisateur en base SQLite,
MLflow pour le tracking, Docker + docker-compose, CI/CD GitHub Actions,
monitoring Prometheus + Grafana + Uptime Kuma.

**Phase 10 — Intégration du modèle final dans l'API.** Intégration du
pipeline XGBoost et de son seuil de décision optimisé — retenu en Phase 7 à
l'issue de la comparaison des trois modèles — dans `src/api/predict.py`.
Ajout d'une interprétation lisible des prédictions (signaux cliniques et
textuels)

## 4. Démonstration d'inférence avec interprétation lisible

On illustre ici le fonctionnement de bout en bout de l'API en appelant directement
la fonction `predict_one` utilisée en production. Le résultat inclut le niveau
d'urgence prédit, les probabilités par classe, et une interprétation en langage
naturel des signaux qui ont influencé la décision.

In [11]:
import sys
from pathlib import Path

# On remonte à la racine du projet pour pouvoir importer src/
ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))

from src.api.predict import predict_one

# Cas 1 : urgence vitale claire
cas_critique = {
    "sexe": "H",
    "age": 71,
    "zone_vie": "U",
    "source": "appel",
    "freq_cardiaque": 122.0,
    "tension_sys": 176.0,
    "temp": 39.2,
    "sat_oxygene": 88.0,
    "antecedents": 1,
    "duree_symptomes": 1.0,
    "description_symptomes": "essoufflement intense et douleur thoracique",
}

resultat = predict_one(cas_critique)

print("=" * 60)
print("CAS 1 — Urgence potentiellement vitale")
print("=" * 60)
print(f"Niveau prédit : {resultat['niveau_urgence']} — {resultat['label']}")
print(f"Probabilités  : {resultat['probabilites']}")
print(f"\nInterprétation :")
for ligne in resultat['interpretation']:
    print(f"  • {ligne}")

CAS 1 — Urgence potentiellement vitale
Niveau prédit : 2 — Urgence vitale ⚠️
Probabilités  : {'non_urgent': 0.0, 'urgence_relative': 0.9605, 'urgence_vitale': 0.0395}

Interprétation :
  • La description mentionne une douleur thoracique, signal souvent associé à un risque élevé.
  • La description mentionne un essoufflement, ce qui peut orienter vers une situation plus urgente.
  • La saturation en oxygène est basse, ce qui augmente le niveau de vigilance.
  • La fréquence cardiaque est anormale, ce qui peut indiquer une situation plus urgente.
  • La température est élevée.
  • La tension systolique est élevée.
  • Probabilités estimées par le modèle : classe 0: 0.0%, classe 1: 96.1%, classe 2: 3.9%. Classe retenue : `2`.
  • Le modèle utilise un seuil plus prudent pour la classe `2` afin de limiter les urgences vitales manquées.


In [12]:
# Cas 2: situation non urgente
cas_non_urgent = {
    "sexe": "F",
    "age": 28,
    "zone_vie": "U",
    "source": "chat",
    "freq_cardiaque": 72.0,
    "tension_sys": 118.0,
    "temp": 36.8,
    "sat_oxygene": 98.0,
    "antecedents": 0,
    "duree_symptomes": 72.0,
    "description_symptomes": "demande de certificat médical pour activité sportive",
}

resultat2 = predict_one(cas_non_urgent)

print("=" * 60)
print("CAS 2 — Situation non urgente")
print("=" * 60)
print(f"Niveau prédit : {resultat2['niveau_urgence']} — {resultat2['label']}")
print(f"Probabilités  : {resultat2['probabilites']}")
print(f"\nInterprétation :")
for ligne in resultat2['interpretation']:
    print(f"  • {ligne}")

CAS 2 — Situation non urgente
Niveau prédit : 0 — Non urgent
Probabilités  : {'non_urgent': 1.0, 'urgence_relative': 0.0, 'urgence_vitale': 0.0}

Interprétation :
  • Probabilités estimées par le modèle : classe 0: 100.0%, classe 1: 0.0%, classe 2: 0.0%. Classe retenue : `0`.


## 5. Limites et perspectives

**Limites actuelles** :

- Pas de validation clinique réelle — le dataset est un jeu de données fourni pour
  l'exercice, pas une cohorte de patients réels validée médicalement.
- TF-IDF ne gère pas la négation ni le contexte sémantique fin (ex: "pas de douleur
  thoracique" pourrait être mal interprété)
- Le seuil de décision abaissé augmente le taux de sur-triage (faux positifs sur
  la classe 2), ce qui est un choix métier assumé mais a un coût opérationnel réel.
- Les logs d'inférence contiennent le texte brut du patient, à chiffrer/anonymiser
  strictement en environnement de production réel

**Découplage texte/label dans le dataset synthétique** (limite identifiée par un
test post-déploiement, au-delà du périmètre du dataset fourni) :

En rejouant ~60 cas réels du jeu de test à travers l'API déployée, j'ai constaté
que certaines phrases de description se retrouvent associées à des labels
différents selon les lignes. Exemple concret :

> *« Plaie par arme blanche au thorax avec détresse circulatoire visible. »*
> — associée à des constantes calmes (FC 82, SpO2 96,5 %) dans une ligne → **Non urgent**
> — associée à des constantes alarmantes dans une autre ligne → **Urgence vitale**

Cela démontre que, dans ce dataset synthétique, **le texte n'est pas causalement
lié au label** — seules les constantes vitales le sont. Le modèle a donc appris,
à raison compte tenu des données, à s'appuyer presque exclusivement sur les
constantes chiffrées et à largement sous-pondérer la sémantique du texte. Le score
F1 d'environ 0.91 obtenu par le texte seul (scénario `text_only`, cf. notebook 03)
provient donc probablement d'une corrélation statistique de surface (fréquence de
certains mots) plutôt que d'une réelle compréhension du sens clinique

Conséquence pratique : sur du texte libre réellement rédigé par un patient ou un
soignant (hors vocabulaire des ~46 phrases-modèles du dataset), le signal textuel
risque d'être largement ignoré par le modèle au profit des seules constantes
vitales, un comportement à anticiper avant tout déploiement sur des données
réelles.

**Perspectives d'amélioration** :

- Remplacer TF-IDF par un modèle de langage français plus riche (CamemBERT) si le
  volume de données et le besoin de nuance le justifient
- Ajouter une gestion explicite de la négation dans le pipeline texte.
- Mettre en place un monitoring de dérive des données (data drift) en production.
- Permettre l'ajustement du seuil de décision directement depuis l'interface
  d'administration, sans réentraînement
- Intégrer une revue manuelle des feedbacks utilisateurs avant réinjection dans
  le jeu d'entraînement. À ce stade, `/retrain` relit uniquement les données
  sources d'origine, les feedbacks collectés via `/feedback` sont stockés et
  consultables, mais pas encore exploités par le réentraînement. C'est un choix
  volontaire, pour garder un contrôle humain sur les données médicales avant
  tout ré-apprentissage, mais c'est aussi l'étape suivante logique pour fermer
  réellement la boucle d'amélioration continue

## Conclusion

Le pipeline **XGBoost avec seuil de décision ajusté (0.01) sur la classe 2** offre
le meilleur compromis entre performance globale et sécurité métier. Il assume une
légère baisse du F1 pondéré pour réduire fortement les erreurs critiques, un choix
cohérent avec un contexte où manquer une urgence vitale est l'erreur à éviter en
priorité